# Learning-to-rank : pyTerrier - OpenNIR

Dans cette partie, on s'intéresse à l'utilisation de modèles neuronaux pour la recherche d'information.
Les modèles neuronaux utilisés ont été rassemblés dans la librairie [OpenNIR](https://opennir.net/).
On explorera également le modèle T5 avec le plugin [monoT5](https://github.com/terrierteam/pyterrier_t5).

In [1]:
# !uv pip install --upgrade python-terrier
# !uv pip install --upgrade git+https://github.com/Georgetown-IR-Lab/OpenNIR
# !uv pip install --upgrade git+https://github.com/terrierteam/pyterrier_t5

## Initialisation
De façon similaire au TP1, on initialise PyTerrier. Nous allons travailler sur le dataset CORD19. le bloc suivant est une répétition du code en TP1.

In [2]:
import pyterrier as pt
if not pt.started():
    pt.init(tqdm='notebook')
import onir_pt

/var/folders/hr/c9xk46kd0t3c2nlgs6zkg2_h0000gn/T/ipykernel_74639/3869899361.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/var/folders/hr/c9xk46kd0t3c2nlgs6zkg2_h0000gn/T/ipykernel_74639/3869899361.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
The following code will have the same effect:
pt.utils.set_tqdm('notebook')
pt.java.init() # optional, forces java initialisation
  pt.init(tqdm='notebook')


Better speed can be achieved with apex installed from https://www.github.com/nvidia/apex.


In [3]:
import os

dataset = pt.datasets.get_dataset('irds:cord19/trec-covid')
topics = dataset.get_topics(variant='description')
qrels = dataset.get_qrels()

indexer = pt.index.IterDictIndexer('./cord19-index', text_attrs=['title', 'abstract'], fields=True)
indexref = indexer.index(dataset.get_corpus_iter())
index = pt.IndexFactory.of(indexref)



cord19/trec-covid documents:   0%|          | 0/192509 s<?, ?it/s]

23:36:30.392 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
23:36:44.344 [ForkJoinPool-1-worker-1] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(In

In [5]:
print(os.path.exists("terrier_index.zip"))

False


In [6]:
#Si le chargement de la collection est trop long à cause du débit ou autres raisons,
# il est possible de récupérer directement l'index fourni par Terrier
import os

if not os.path.exists("terrier_index.zip"):
  !wget http://www.dcs.gla.ac.uk/~craigm/ecir2021-tutorial/terrier_index.zip
  !unzip -j terrier_index.zip -d terrier_index

index_ref = pt.IndexRef.of("./terrier_index/data.properties")
index = pt.IndexFactory.of(index_ref)

--2026-05-08 23:37:35--  http://www.dcs.gla.ac.uk/~craigm/ecir2021-tutorial/terrier_index.zip
Resolving www.dcs.gla.ac.uk (www.dcs.gla.ac.uk)... 130.209.240.1
Connecting to www.dcs.gla.ac.uk (www.dcs.gla.ac.uk)|130.209.240.1|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dcs.gla.ac.uk/~craigm/ecir2021-tutorial/terrier_index.zip [following]
--2026-05-08 23:37:36--  https://www.dcs.gla.ac.uk/~craigm/ecir2021-tutorial/terrier_index.zip
Connecting to www.dcs.gla.ac.uk (www.dcs.gla.ac.uk)|130.209.240.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42017186 (40M) [application/zip]
Saving to: ‘terrier_index.zip’

terrier_index.zip   100%[===================>]  40.07M  1.55MB/s    in 16s     

2026-05-08 23:37:52 (2.49 MB/s) - ‘terrier_index.zip’ saved [42017186/42017186]

Archive:  terrier_index.zip
  inflating: terrier_index/data.lexicon.fsomapfile  
  inflating: terrier_index/data.properties  
  inflating: terrier_index/

## Modèles de ré-ordonancement neuronaux "from scratch"

Les modèles de ré-ordonnancement dans OpenNIR sont constitués de deux éléments :
*  ranker: un modèle d'ordonnancement (e.g., drmm, knrm, pacrr, ...). Cf la liste des [Rankers](https://opennir.net/rankers.html). Il est également possible de rajouter des modèles d'ordonnancement en étendant la classe Ranker.
*  vocab : défnit comment le texte est encodé par le modèle (e.g., wordvec_hash, bert, ...). Cela permet ainsi de tester plusieurs méthodes de représentation. Plus de détails concernant le [vocab](https://opennir.net/vocab.html).

Les modèles de réordonnancement s'appuient sur une première étape d'ordonnancement (souvent BM25), récupèrent ensuite les textes des top documents et appliquent ensuite le modèle neuronal.



In [7]:
knrm = onir_pt.reranker('knrm', 'wordvec_hash', text_field='abstract')


config file not found: config
[2026-05-08 23:40:38,023][WordvecHashVocab][DEBUG] [starting] downloading https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip


[2026-05-08 23:43:28,012][onir.util.download][WARNING] no hash provided for https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip; consider adding expected_md5="3cc8839ac3fa9a6187149b1e73328b2a" to ensure data integrity.
[2026-05-08 23:43:28,014][onir.util.download][DEBUG] downloaded https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip [02:49] [682M] [2.73MB/s]
[2026-05-08 23:43:28,018][WordvecHashVocab][DEBUG] [finished] downloading https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip [02:50]
[2026-05-08 23:43:28,019][WordvecHashVocab][DEBUG] [starting] extracting vecs
[2026-05-08 23:43:30,525][WordvecHashVocab][DEBUG] [finished] extracting vecs [2.51s]
[2026-05-08 23:43:30,527][WordvecHashVocab][DEBUG] [starting] loading vecs into memory
[2026-05-08 23:44:51,274][WordvecHashVocab][DEBUG] [finished] loading vecs into memory [01:21]
[2026-05-08 23:44:51,466][WordvecHashVocab][DEBUG] [starting] writing

Une fois le modèle chargé, il est nécessaire de mettre en place la pipeline d'évaluation vue dans le tp1.
Le modèle neuronal n'est pas efficace car il n'a pas été entraîné et utilise des poids aléatoires.

In [8]:
br = pt.BatchRetrieve(index) % 100
pipeline = br >> pt.text.get_text(dataset, 'abstract') >> knrm
pt.Experiment(
    [br, pipeline],
    topics,
    qrels,
    names=['DPH', 'DPH >> KNRM'],
    eval_metrics=["map", "ndcg", 'ndcg_cut.10', 'P.10', 'mrt']
)

/var/folders/hr/c9xk46kd0t3c2nlgs6zkg2_h0000gn/T/ipykernel_74639/3806215551.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  br = pt.BatchRetrieve(index) % 100


[2026-05-08 23:45:29,918][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-08 23:45:29,924][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-05-08 23:45:29,932][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/Users/vlad/Documents/University/Master-MIND/M1-S2-MIND/RITAL/tme/tme07/.venv/lib/python3.13/site-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: DPH >> KNRM ((TerrierRetr(DPH) >> RankCutoff(100) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x14f60da90> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-05-08 23:45:32,455][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-08 23:45:32,455][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/1250 s<?, ?it/s]

[2026-05-08 23:45:33,449][onir_pt][DEBUG] [finished] batches: s] [1250it] [1257.47it/s]


,name,map,P.10,ndcg,ndcg_cut.10,mrt
0,DPH,0.068056,0.658,0.165653,0.609058,1382.538417
1,DPH >> KNRM,0.054809,0.450,0.145486,0.359992,2078.396042


## Entraînement des modèles sur les jeux de données

Pour entraîner les modèles, il est nécessaire de construire la pipeline de modèles/transformations à utliser et ensuite d'appliquer la fonction .fit() à la pipeline.
Le code ci-dessous prend beaucoup de temps. Il est donné à titre indicatif si vous souhaitez l'utiliser sur des serveurs adaptés. L'étape suivante permet de charger directement les poids du modèle pré-entraîné à l'avance et mis à disposition de la communauté.

Dans ce qui suit, on utilise le jeu de données MS MARCO medical pour pré-entraîner le modèle qui sera ensuite appliqué sur CORD19.
Il est également possible de garder seulement le jeu de données CORD19, de le découper en train/val/test et regarder les performances.

In [ ]:
# Apprentissage du modèle sur des données médicales (MS MARCO medical)
from sklearn.model_selection import train_test_split
train_ds = pt.datasets.get_dataset('irds:msmarco-passage/train/medical')
train_topics, valid_topics = train_test_split(train_ds.get_topics(), test_size=50, random_state=42)

# Indexation de MS MARCO pour la première étape d'ordonnancement et récupérer les textes (pour le ranker openNIR)
indexer = pt.index.IterDictIndexer(
    './terrier_msmarco-passage',
    text_attrs=['text'],
    meta={'docno': 20} # Specify 'docno' and its maximum byte length here
)
tr_index_ref = indexer.index(train_ds.get_corpus_iter())

pipeline = (pt.BatchRetrieve(tr_index_ref) % 100 # récupère les 100 premiers documents
            >> pt.text.get_text(train_ds, 'text') # récupère le texte de ces documents
            >> pt.apply.generic(lambda df: df.rename(columns={'text': 'abstract'})) # renomme la colonne
            >> knrm) # applique le re-ranker


pipeline.fit(
    train_topics,
    train_ds.get_qrels(),
    valid_topics,
    train_ds.get_qrels())

Pour éviter l'étape d'entraînement ici, on utlise une version pré-entraînée.

**Important** : penser à supprimer la mémoire des re-rankers qui vont être mis à jour avec les modèles pré-entrainés.

In [17]:
del knrm # free up the memory before loading a new version of the ranker
knrm = onir_pt.reranker.from_checkpoint('https://macavaney.us/knrm.medmarco.tar.gz', text_field='abstract', expected_md5="d70b1d4f899690dae51161537e69ed5a")

[2026-05-08 23:58:02,012][onir_pt][INFO] using cached checkpoint: /Users/vlad/data/onir/model_checkpoints/b7694d2fb4d4f8218e11734de239ad30
[2026-05-08 23:58:02,013][WordvecHashVocab][DEBUG] [starting] reading cached at /Users/vlad/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p
[2026-05-08 23:58:02,786][WordvecHashVocab][DEBUG] [finished] reading cached at /Users/vlad/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p s]


Il est ensuire possible de lancer la pipeline d'évaluation avec ce nouveau modèle. Les résultats sont meilleurs !

In [18]:
pipeline = br >> pt.text.get_text(dataset, 'abstract') >> knrm
pt.Experiment(
    [br, pipeline],
    topics,
    qrels,
    names=['DPH', 'DPH >> KNRM'],
    baseline=0,       ## spécifie quelle est le modèle de référence pour calculer les améliorations.
    eval_metrics=["map", "ndcg", 'ndcg_cut.10', 'P.10', 'mrt']
)

[2026-05-08 23:58:10,275][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-08 23:58:10,276][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-05-08 23:58:10,281][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/Users/vlad/Documents/University/Master-MIND/M1-S2-MIND/RITAL/tme/tme07/.venv/lib/python3.13/site-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: DPH >> KNRM ((TerrierRetr(DPH) >> RankCutoff(100) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x15871f360> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-05-08 23:58:12,236][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-08 23:58:12,237][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/1250 s<?, ?it/s]

[2026-05-08 23:58:13,128][onir_pt][DEBUG] [finished] batches: s] [1250it] [1402.31it/s]


,name,map,P.10,ndcg,ndcg_cut.10,mrt,map +,map -,map p-value,P.10 +,P.10 -,P.10 p-value,ndcg +,ndcg -,ndcg p-value,ndcg_cut.10 +,ndcg_cut.10 -,ndcg_cut.10 p-value
0,DPH,0.068056,0.658,0.165653,0.609058,888.664500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DPH >> KNRM,0.065103,0.598,0.160561,0.532655,1915.463833,20.0,30.0,0.095852,12.0,26.0,0.024604,20.0,30.0,0.028273,20.0,30.0,0.005972


**Exercice 1**
Autre pipeline. A vous de deviner ce qu'elle fait ! 

In [19]:
cutoffs = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
dph = pt.BatchRetrieve(index)
res = pt.Experiment(
    [dph % cutoff >> pt.text.get_text(dataset, 'abstract') >> knrm for cutoff in cutoffs],
    dataset.get_topics('description'),
    dataset.get_qrels(),
    names=[f'c={cutoff}' for cutoff in cutoffs],
    eval_metrics=["map", "recip_rank", "ndcg", "ndcg_cut.10", "mrt"]
)
res

[2026-05-08 23:59:13,625][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-08 23:59:13,626][onir_pt][DEBUG] [starting] batches


/var/folders/hr/c9xk46kd0t3c2nlgs6zkg2_h0000gn/T/ipykernel_74639/2093709227.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  dph = pt.BatchRetrieve(index)


batches: 0it s, ?it/s]

[2026-05-08 23:59:13,629][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]
[2026-05-08 23:59:13,638][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-08 23:59:13,638][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

...

batches:   0%|          | 0/1125 s<?, ?it/s]

[2026-05-08 23:59:26,594][onir_pt][DEBUG] [finished] batches: s] [1125it] [1301.48it/s]
[2026-05-08 23:59:27,479][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-08 23:59:27,479][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/1250 s<?, ?it/s]

[2026-05-08 23:59:28,383][onir_pt][DEBUG] [finished] batches: s] [1250it] [1383.52it/s]


,name,map,recip_rank,ndcg,ndcg_cut.10,mrt
0,c=10,0.011830,0.785333,0.049378,0.596982,1079.600500
1,c=20,0.020943,0.787500,0.071904,0.587704,1027.257375
2,c=30,0.029340,0.805667,0.089954,0.598276,1150.044583
3,c=40,0.034921,0.808524,0.103086,0.582889,1308.777833
4,c=50,0.040708,0.825667,0.114785,0.591845,1246.147333
5,c=60,0.046930,0.815667,0.126095,0.589046,1414.800167
6,c=70,0.051907,0.816333,0.136294,0.578130,1608.761292
7,c=80,0.056873,0.804667,0.145013,0.565053,1808.077792
8,c=90,0.061703,0.806333,0.154460,0.569579,1806.505500
9,c=100,0.065103,0.767889,0.160561,0.532655,1745.137209


So we just varied cutoffs before reranking.

## Modèle Vanilla BERT

**Exercice 2**

Sur le même principe que le modèle KNRM, analyser les performances du modèle vanilla BERT sans et avec pré-entraînement.
Pour la version du modèle pré-entraîné, on utilisera le checkpoint ['https://macavaney.us/scibert-medmarco.tar.gz']('https://macavaney.us/scibert-medmarco.tar.gz') avec le paramètre expected_md5="854966d0b61543ffffa44cea627ab63b".

Synthétisez toutes les mesures d'évaluation dans un même tableau (bm25, knrm et Vanilla Bert / avec/sans entraînement).

In [ ]:
# vanilla_bert = onir_pt.reranker(
#     "vanilla_bert", "wordvec_hash", text_field="abstract",
# ) # TODO: doesn't load ;(

In [25]:
vanilla_bert = onir_pt.reranker.from_checkpoint(
    "https://macavaney.us/scibert-medmarco.tar.gz",
    text_field="abstract",
    expected_md5="854966d0b61543ffffa44cea627ab63b",
)

[2026-05-09 00:19:36,046][onir.util.download][DEBUG] downloaded https://macavaney.us/scibert-medmarco.tar.gz [02:14] [499M] [10.4MB/s] [md5 hash verified]


[2026-05-09 00:24:05,458][onir.util.download][DEBUG] downloaded https://s3-us-west-2.amazonaws.com/ai2-s2-research/scibert/pytorch_models/scibert_scivocab_uncased.tar [04:28] [411M] [2.42MB/s] [md5 hash verified]


extracting: 411MB s, 1.00GB/s]
extracting: 821MB [1.92s, 428MB/s] 


In [26]:
pipeline = br >> pt.text.get_text(dataset, 'abstract') >> knrm
pt.Experiment(
    [br, pipeline],
    topics,
    qrels,
    names=['DPH', 'DPH >> BERT'],
    baseline=0,       ## spécifie quelle est le modèle de référence pour calculer les améliorations.
    eval_metrics=["map", "ndcg", 'ndcg_cut.10', 'P.10', 'mrt']
)

[2026-05-09 00:26:20,861][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-09 00:26:20,862][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-05-09 00:26:20,867][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/Users/vlad/Documents/University/Master-MIND/M1-S2-MIND/RITAL/tme/tme07/.venv/lib/python3.13/site-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: DPH >> BERT ((TerrierRetr(DPH) >> RankCutoff(100) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x17c7bc6d0> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-05-09 00:26:23,179][onir_pt][ERROR] gpu=True, but CUDA is not available. Falling back on CPU.
[2026-05-09 00:26:23,180][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/1250 s<?, ?it/s]

[2026-05-09 00:26:24,365][onir_pt][DEBUG] [finished] batches: [1.18s] [1250it] [1055.27it/s]


,name,map,P.10,ndcg,ndcg_cut.10,mrt,map +,map -,map p-value,P.10 +,P.10 -,P.10 p-value,ndcg +,ndcg -,ndcg p-value,ndcg_cut.10 +,ndcg_cut.10 -,ndcg_cut.10 p-value
0,DPH,0.068056,0.658,0.165653,0.609058,1181.8685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DPH >> BERT,0.065103,0.598,0.160561,0.532655,2268.6565,20.0,30.0,0.095852,12.0,26.0,0.024604,20.0,30.0,0.028273,20.0,30.0,0.005972


# Autres modèles - hors OpenNIR
PyTerrier a mis a disposition l'implémentation d'autres modèles neuronaux:
- ColBERT : https://github.com/terrierteam/pyterrier_colbert
- T5 : https://github.com/terrierteam/pyterrier_t5
- Doc2Query : https://github.com/terrierteam/pyterrier_doc2query
- DeepCT : https://github.com/terrierteam/pyterrier_deepct
- ANCE : https://github.com/terrierteam/pyterrier_ance

Un exemple de code de code est donnée ci-dessous avec Mono-T5 :

In [27]:
from pyterrier_t5 import MonoT5ReRanker
monoT5 = MonoT5ReRanker(text_field='abstract')

br = pt.BatchRetrieve(index) % 30
pipeline = (br >> pt.text.get_text(dataset, 'abstract') >> monoT5)
pt.Experiment(
    [br, pipeline],
    dataset.get_topics('description'),
    dataset.get_qrels(),
    names=['DPH', 'DPH >> T5'],
    eval_metrics=["map", "recip_rank", "ndcg", "ndcg_cut.10", "mrt"]
)

spiece.model:   0%|          | 0.00/792k s<?, ?B/s]

tokenizer.json: 0.00B s, ?B/s]

config.json: 0.00B s, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M s<?, ?B/s]

Loading weights:   0%|          | 0/260 s<?, ?it/s]

/var/folders/hr/c9xk46kd0t3c2nlgs6zkg2_h0000gn/T/ipykernel_74639/797270371.py:4: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  br = pt.BatchRetrieve(index) % 30


model.safetensors:   0%|          | 0.00/892M s<?, ?B/s]

monoT5:   0%|          | 0/375 s<?, ?batches/s]

,name,map,recip_rank,ndcg,ndcg_cut.10,mrt
0,DPH,0.029856,0.829190,0.091589,0.609058,813.022917
1,DPH >> T5,0.032264,0.860667,0.095550,0.709075,167426.032083
